# SCBE Stack-Agent Staged Training

Train a code agent on the SCBE stack action grammar: inspect, run bounded checks, repair from evidence, and emit receipts.

This notebook is staged and artifact-first. The repo data remains the source of truth; Colab is only compute.


In [ ]:
# Stage 0: configuration
import os, json, pathlib, subprocess, textwrap

SCBE_REPO = os.environ.get('SCBE_REPO', 'https://github.com/issdandavis/SCBE-AETHERMOORE.git')
SCBE_REF = os.environ.get('SCBE_REF', 'feat/stack-agent-training-lane')
BASE_MODEL = os.environ.get('BASE_MODEL', 'Qwen/Qwen2.5-Coder-3B-Instruct')
LR = float(os.environ.get('STACK_AGENT_LR', '5e-5'))
EPOCHS = float(os.environ.get('STACK_AGENT_EPOCHS', '1'))
MAX_LEN = int(os.environ.get('STACK_AGENT_MAX_LEN', '2048'))
RUN_CODE_EVAL = os.environ.get('RUN_CODE_EVAL', '1') == '1'
ROOT = pathlib.Path('/content/scbe')
STAGE = pathlib.Path('/content/stack_agent_stage')
STAGE.mkdir(parents=True, exist_ok=True)
print({'repo': SCBE_REPO, 'ref': SCBE_REF, 'base_model': BASE_MODEL, 'lr': LR, 'epochs': EPOCHS, 'stage': str(STAGE)})


In [ ]:
# Stage 1: checkout source of truth
import shutil, os, subprocess, pathlib
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', SCBE_REF, SCBE_REPO, str(ROOT)], check=True)
os.chdir(ROOT)
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# Stage 2: install training dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.44.0', 'datasets', 'accelerate', 'peft', 'bitsandbytes'], check=True)


In [ ]:
# Stage 3: build and gate the stack-agent corpus
import subprocess, sys, json, pathlib
corpus = STAGE / 'stack_agent_seed.jsonl'
manifest = STAGE / 'stack_agent_seed.manifest.json'
report = STAGE / 'stack_agent_eval_report.json'
subprocess.run([sys.executable, 'scripts/training/build_stack_repair_corpus.py', '--out', str(corpus), '--manifest', str(manifest), '--include-smoke'], check=True)
subprocess.run([sys.executable, 'scripts/training/eval_stack_agent.py', '--corpus', str(corpus), '--out', str(report)], check=True)
print(manifest.read_text()[:1000])
print(report.read_text()[:1000])


In [ ]:
# Stage 4: load model and tokenize corpus
import json, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'))

def render_chat(row):
    messages = row['messages']
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return '
'.join(f"{m['role'].upper()}: {m['content']}" for m in messages)

ds = load_dataset('json', data_files=str(corpus), split='train')

def tok(row):
    text = render_chat(row)
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, padding='max_length')
    enc['labels'] = enc['input_ids'].copy()
    return enc

tok_ds = ds.map(tok, remove_columns=ds.column_names)
print({'records': len(tok_ds), 'max_len': MAX_LEN})


In [ ]:
# Stage 5: gentle adapter training
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import json, pathlib

out_dir = STAGE / 'adapter'
args = TrainingArguments(
    output_dir=str(out_dir),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=1,
    save_strategy='epoch',
    report_to=[],
    fp16=True,
)
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
train_result = trainer.train()
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
metrics = dict(train_result.metrics)
metrics.update({'lr': LR, 'epochs': EPOCHS, 'base_model': BASE_MODEL, 'records': len(tok_ds)})
(STAGE / 'train_metrics.json').write_text(json.dumps(metrics, indent=2, sort_keys=True))
print(metrics)


In [ ]:
# Stage 6: optional held-out pitfall code-lift eval
# This is the honest capability ruler. It may be slow; disable with RUN_CODE_EVAL=0.
import json, re, torch
from python.helm import pitfall_eval
from python.helm.code_lift import solve_rate, lift_from_solve, render

problems = pitfall_eval.eval_problems()

def extract_code(text):
    m = re.search(r'```(?:python)?\n(.*?)```', text, re.S)
    return (m.group(1) if m else text).strip()

def make_prompt(p):
    return 'Write a complete Python solution. Return only code.\n\n' + p['text'] + '\n\nVisible test:\n' + p['test_list'][0]

def gen_one(p):
    inputs = tokenizer(make_prompt(p), return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=384, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return extract_code(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

if RUN_CODE_EVAL:
    model.eval()
    with model.disable_adapter():
        base = solve_rate(problems, gen_one, public_k=1)
    trained = solve_rate(problems, gen_one, public_k=1)
    rep = lift_from_solve(base, trained)
    serial = {k: (sorted(v) if isinstance(v, set) else v) for k, v in rep.items()}
    (STAGE / 'report.json').write_text(json.dumps(serial, indent=2, sort_keys=True))
    print(render(rep))
else:
    (STAGE / 'report.json').write_text(json.dumps({'code_eval_skipped': True}, indent=2))
    print('code eval skipped')


In [ ]:
# Stage 7: artifact manifest
import json, pathlib, os
artifacts = sorted(str(p) for p in STAGE.glob('**/*') if p.is_file())
summary = {
    'status': 'STACK_AGENT_TRAINING_DONE',
    'stage': str(STAGE),
    'artifacts': artifacts,
}
(STAGE / 'artifact_manifest.json').write_text(json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(summary, indent=2)[:4000])
